In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import r2_score
import scipy.stats as stats
from sklearn.preprocessing import PowerTransformer
from sklearn.linear_model import LinearRegression

In [18]:
df=pd.read_csv('concrete_data.csv')
df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [19]:
df.isnull().sum()

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

In [20]:
df.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [21]:
X=df.drop(columns=['Strength'])
y=df["Strength"]

In [22]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [24]:
#### applying linear regression  ###

lr=LinearRegression()
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)
r2_score(y_test,y_pred)

0.6275531792314851

In [ ]:
###Cross-checking with cross-val-score ####

### cross validation calculates the percentage on how much a model perform well on unseen data

lr=LinearRegression()
np.mean(cross_val_score(lr,X,y,scoring="r2"))

np.float64(0.4609940491662865)

In [32]:
###  Applying box-cox transformation ####
pt=PowerTransformer(method='box-cox')
X_train_transformed=pt.fit_transform(X_train+0.0000001)
X_test_transformed=pt.transform(X_test+0.000000001)

pd.DataFrame(({'cols':X_train.columns,'box-cox_lamdas':pt.lambdas_}))

# 2️⃣ pt.lambdas_

# pt is your PowerTransformer object (method='box-cox' or 'yeo-johnson')

# After you call fit or fit_transform on X_train, it calculates λ (lambda) for each feature

# λ is the power parameter used to transform that feature


# here the data is normally distributed by lamda

,cols,box-cox_lamdas
0,Cement,0.177025
1,Blast Furnace Slag,0.022767
2,Fly Ash,-0.034635
3,Water,0.772682
4,Superplasticizer,0.087141
5,Coarse Aggregate,1.129813
6,Fine Aggregate,1.782018
7,Age,0.066631


In [33]:
### Applying linear Regression ####
##so transforming make the inputation normal distrubuted according to which the y should be predicted to its original value what it should need to give the output
lr=LinearRegression()
lr.fit(X_train_transformed,y_train)
y_pred2=lr.predict(X_test_transformed)
r2_score(y_test,y_pred2)

0.7922243572066714

In [38]:
#### using cross-validation ### 
pt = PowerTransformer(method='box-cox')
X_transformed = pt.fit_transform(X + 0.000001)

lr = LinearRegression()
np.mean(cross_val_score(lr, X_transformed, y, scoring='r2'))

np.float64(0.6662950327179044)

In [37]:
scores = cross_val_score(lr, X_train_transformed, y_train, scoring="r2", cv=5)
print("Mean R2:", np.mean(scores))

Mean R2: 0.7922924851884379


In [40]:
### Applying yeo_johnson  ####


pt1=PowerTransformer()
X_train_transformed1=pt1.fit_transform(X_train)
X_test_transformed1=pt1.transform(X_test)

pd.DataFrame(({'cols':X_train.columns,'box-cox_lamdas':pt1.lambdas_}))



,cols,box-cox_lamdas
0,Cement,0.174348
1,Blast Furnace Slag,0.015715
2,Fly Ash,-0.161447
3,Water,0.771307
4,Superplasticizer,0.253935
5,Coarse Aggregate,1.130050
6,Fine Aggregate,1.783100
7,Age,0.019885


In [41]:
lr=LinearRegression()
lr.fit(X_train_transformed1,y_train)
y_pred_3=lr.predict(X_test_transformed1)
r2_score(y_test,y_pred_3)

0.8161906513354853

In [43]:
###  cross-validation  ##
scores = cross_val_score(lr, X_train_transformed1, y_train, scoring="r2", cv=5)
print("Mean R2:", np.mean(scores))

Mean R2: 0.7941782182452176
